In [1]:
!pip install -q bm25s
!pip install -q bm25-vectorizer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 940.8 kB/s eta 0:00:000:00:01


In [2]:
import os
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [3]:
CONFIG = {
    "dataset": "amazon",  # Switch to "sentiment140" to test adaptability
    "paths": {
        "amazon_raw": "/kaggle/input/datasets/organizations/snap/amazon-fine-food-reviews/Reviews.csv",
        "sentiment140_raw": "/kaggle/input/datasets/kazanova/sentiment140/training.1600000.processed.noemoticon.csv"
    },
    "preprocessing": {
        "lowercase": True,
        "remove_stopwords": True,
        "method": "lemmatization",  # Options: "stemming", "lemmatization", "none"
        "handle_social_tokens": True,  # New: Converts @mentions and URLs
        "expand_contractions": True,   # New: Fixes don't -> do not
        "strip_html": True,            # New: Cleans Amazon artifacts
        "collapse_elongated": True     # New: loooove -> loove
    }
}

In [4]:
# --- DATASET CONFIGURATIONS (Future config/data/amazon.yaml) ---
AMAZON_CONFIG = {
    "dataset_name": "amazon",
    "raw_path": "/kaggle/input/datasets/organizations/snap/amazon-fine-food-reviews/Reviews.csv",
    "read_csv_kwargs": {},  # Standard loading defaults
    "column_mapping": {
        "Text": "raw_text", 
        "Score": "target"
    },
    "preprocessing": {
        "strip_html": True,
        "handle_social_tokens": False,
        "expand_contractions": False,
        "remove_stopwords": True,
        "max_features": 15000
    },
    # Handled inside the ingestion step:
    "drop_classes": [3],  # Drop neutral reviews
    "target_mapping": {1: 0, 2: 0, 4: 1, 5: 1} # 1-2 -> Neg(0), 4-5 -> Pos(1)
}

# --- DATASET CONFIGURATIONS (Future config/data/sentiment140.yaml) ---
SENTIMENT140_CONFIG = {
    "dataset_name": "sentiment140",
    "raw_path": "/kaggle/input/datasets/kazanova/sentiment140/training.1600000.processed.noemoticon.csv",
    "read_csv_kwargs": {
        "encoding": "latin-1", 
        "header": None
    },
    "column_mapping": {
        5: "raw_text", 
        0: "target"
    },
    "preprocessing": {
        "strip_html": False,
        "handle_social_tokens": True,
        "remove_stopwords": False,
        "expand_contractions": True,
        "max_features": 25000
    },
    # Handled inside the ingestion step:
    "drop_classes": [],  # Nothing to drop
    "target_mapping": {0: 0, 4: 1} # 0 -> Neg(0), 4 -> Pos(1)
}

In [5]:
pd.read_csv(CONFIG["paths"]["sentiment140_raw"], encoding="latin-1", header=None)[0].value_counts()

0
0    800000
4    800000
Name: count, dtype: int64

In [6]:
import pandas as pd

def load_raw_dataset(data_config: dict, nrows: int = None) -> pd.DataFrame:
    """
    Loads, standardizes, filters, and encodes any dataset dynamically 
    using its independent configuration properties.
    """
    path = data_config["raw_path"]
    print(f"Loading dataset [{data_config['dataset_name']}] from: {path}")

    # 1. Read file using dataset-specific engine keywords (like encoding/headers)
    read_kwargs = data_config["read_csv_kwargs"].copy()
    if nrows:
        read_kwargs["nrows"] = nrows 
        
    df = pd.read_csv(path, **read_kwargs)

    # 2. Rename columns using configuration maps
    df = df.rename(columns=data_config["column_mapping"])

    # 3. Filter down to relevant columns immediately
    df = df[["raw_text", "target"]].copy()

    # 4. Handle Target Filtering (e.g., Drop neutral 3-star reviews if specified)
    if data_config["drop_classes"]:
        initial_shape = df.shape[0]
        df = df[~df["target"].isin(data_config["drop_classes"])].copy()
        print(f"   Dropped {initial_shape - df.shape[0]} ambiguous rows.")

    # 5. Handle Target Encoding (Map labels to standard 0 and 1)
    print(f"   Applying target sentiment encoding mapping...")
    df["label"] = df["target"].map(data_config["target_mapping"])
        
    # Drop rows that didn't match the mapping schema 
    df = df.dropna(subset=["label"]).copy()
    df["label"] = df["label"].astype(int)

    return df[["raw_text", "label"]]

In [7]:
import re
import html
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

class TextNormalizer:
    def __init__(self, preprocessing_config: dict = None):
        """
        Initializes the normalizer using a configuration dictionary.
        Falls back to standard defaults if keys are missing.
        """
        # If no config is passed, default to an empty dictionary
        cfg = preprocessing_config if preprocessing_config is not None else {}

        # Read parameters from config dictionary with safe fallbacks (.get)
        self.lowercase = cfg.get("lowercase", True)
        self.remove_stopwords = cfg.get("remove_stopwords", True)
        self.method = cfg.get("method", "lemmatization")
        self.handle_social_tokens = cfg.get("handle_social_tokens", False)
        self.expand_contractions = cfg.get("expand_contractions", False)
        self.strip_html = cfg.get("strip_html", False)
        self.collapse_elongated = cfg.get("collapse_elongated", False)

        self.stop_words = set(stopwords.words('english'))
        self.stemmer = PorterStemmer()
        self.lemmatizer = WordNetLemmatizer()

        self.contraction_map = {
            "can't": "cannot", "won't": "will not", "idk": "i do not know", 
            "dont": "do not", "cant": "cannot", "im": "i am"
        }

    def _clean_single_text(self, text: str) -> str:
        if not isinstance(text, str):
            return ""
            
        # 1. HTML Stripping (Prioritize before regexes)
        if self.strip_html:
            text = html.unescape(text)
            text = re.sub(r'<.*?>', ' ', text)
            
        # 2. Handle Social Tokens (Twitter specific)
        if self.handle_social_tokens:
            text = re.sub(r'@\S+', '[USER]', text)
            text = re.sub(r'https?://\S+|www\.\S+', '[URL]', text)
            
        # 3. Collapse elongated words (e.g., coool -> cool)
        if self.collapse_elongated:
            text = re.sub(r'(.)\1+', r'\1\1', text)
            
        if self.lowercase:
            text = text.lower()
            
        # Strip general punctuation here if necessary, but keep structure for social tokens
        if not self.handle_social_tokens:
            text = re.sub(r'[^a-zA-Z\s]', '', text)
            
        # 4. Tokenize and execute standard mappings
        tokens = text.split()
        
        if self.expand_contractions:
            tokens = [self.contraction_map.get(word, word) for word in tokens]
            
        if self.remove_stopwords:
            tokens = [word for word in tokens if word not in self.stop_words]
            
        # 5. Normalization Method (Stemming vs Lemmatization vs None)
        if self.method == "lemmatization":
            tokens = [self.lemmatizer.lemmatize(word) for word in tokens]
        elif self.method == "stemming":
            tokens = [self.stemmer.stem(word) for word in tokens]
            
        return " ".join(tokens)

    def transform(self, texts: pd.Series) -> list:
        print(f"Running text normalization pipeline...")
        return [self._clean_single_text(text) for text in texts]

In [8]:
import bm25s
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

def vectorize_data(cleaned_corpus: list, method: str, **kwargs):
    """
    Standardized factory function for text representations.
    """
    # FIX: Safely extract max_features right here so it is defined globally within this function scope
    max_features = kwargs.get("max_features", 5000)
    
    print(f"Vectorizing corpus using method: {method}...")

    if method == "bow":
        vectorizer = CountVectorizer(max_features=max_features)
        X_matrix = vectorizer.fit_transform(cleaned_corpus)
        return X_matrix, vectorizer

    elif method == "tfidf":
        vectorizer = TfidfVectorizer(max_features=max_features)
        X_matrix = vectorizer.fit_transform(cleaned_corpus)
        return X_matrix, vectorizer

    elif method == "bm25_features":
        # This print statement will now find 'max_features' successfully!
        print(f"Constructing static BM25 feature matrix (max_features={max_features})...")
        tfidf = TfidfVectorizer(max_features=max_features, norm=None, smooth_idf=False)
        X_tfidf = tfidf.fit_transform(cleaned_corpus)
        
        k1 = kwargs.get("k1", 1.5)
        b = kwargs.get("b", 0.75)
        
        doc_lens = X_tfidf.sum(axis=1).A1
        avg_doc_len = doc_lens.mean() if len(doc_lens) > 0 else 1.0
        
        X_bm25 = csr_matrix(X_tfidf.copy())
        
        for i in range(X_bm25.shape[0]):
            start, end = X_bm25.indptr[i], X_bm25.indptr[i+1]
            tf = X_bm25.data[start:end]
            len_norm = 1.0 - b + b * (doc_lens[i] / avg_doc_len)
            X_bm25.data[start:end] = (tf * (k1 + 1)) / (tf + k1 * len_norm)
            
        return X_bm25, tfidf

    else:
        raise ValueError(f"Unknown vectorization method: {method}")

## Load Amazon Data

In [9]:
from sklearn.model_selection import train_test_split
# 1. Load data via your custom ingestion factory
df_amazon = load_raw_dataset(AMAZON_CONFIG)

df_amazon_train, df_amazon_val = train_test_split(
    df_amazon, 
    test_size=0.2, 
    random_state=42, 
    stratify=df_amazon["label"]
)


# 2. Initialize the normalizer passing the nested preprocessing dictionary block
amazon_normalizer = TextNormalizer(preprocessing_config=AMAZON_CONFIG["preprocessing"])

Loading dataset [amazon] from: /kaggle/input/datasets/organizations/snap/amazon-fine-food-reviews/Reviews.csv
   Dropped 42640 ambiguous rows.
   Applying target sentiment encoding mapping...


In [10]:
print("\nTransforming training text data...")
X_amazon_train_clean = amazon_normalizer.transform(df_amazon_train["raw_text"])

print("Transforming validation text data...")
X_amazon_val_clean = amazon_normalizer.transform(df_amazon_val["raw_text"])


Transforming training text data...
Running text normalization pipeline...
Transforming validation text data...
Running text normalization pipeline...


In [11]:
amazon_max_feats = AMAZON_CONFIG["preprocessing"]["max_features"]

X_amazon_train_matrix, trained_backbone = vectorize_data(
    X_amazon_train_clean, 
    method="bm25_features", 
    max_features=amazon_max_feats
)

X_amazon_val_matrix = trained_backbone.transform(X_amazon_val_clean)

amazon_y_train = df_amazon_train["label"].values
amazon_y_val = df_amazon_val["label"].values

Vectorizing corpus using method: bm25_features...
Constructing static BM25 feature matrix (max_features=15000)...


In [13]:
# 1. Choose method from your config and execute
amazon_X_train_tfidf, trained_vectorizer = vectorize_data(X_amazon_train_clean, method="tfidf", max_features=amazon_max_feats)


amazon_X_val_tfidf = trained_vectorizer.transform(X_amazon_val_clean)
# 2. View its mathematical type
print(f"Type of matrix: {type(amazon_X_train_tfidf)}")
print(f"Matrix shape: {amazon_X_train_tfidf.shape} (Rows: Documents, Columns: Unique Words)")

# 3. View the learned vocabulary words
vocab = trained_vectorizer.get_feature_names_out()
# print(f"Top Features Extracted: {list(vocab)}")

Vectorizing corpus using method: tfidf...
Type of matrix: <class 'scipy.sparse._csr.csr_matrix'>
Matrix shape: (420651, 15000) (Rows: Documents, Columns: Unique Words)


In [14]:
trained_vectorizer.get_feature_names_out()

array(['aa', 'aaa', 'aafco', ..., 'zucchini', 'zuke', 'zukes'],
      shape=(15000,), dtype=object)

## Loading Sentiment Dataset

In [15]:
# 1. Load data
df_tweets = load_raw_dataset(SENTIMENT140_CONFIG)

df_tweets_train, df_tweets_val = train_test_split(
    df_tweets, 
    test_size=0.2, 
    random_state=42, 
    stratify=df_tweets["label"]
)

# 2. Initialize the normalizer passing the nested preprocessing dictionary block
twitter_normalizer = TextNormalizer(preprocessing_config=SENTIMENT140_CONFIG["preprocessing"])

print("\nTransforming training text data...")
X_tweets_train_clean = twitter_normalizer.transform(df_tweets_train["raw_text"])

print("Transforming validation text data...")
X_tweets_val_clean = twitter_normalizer.transform(df_tweets_val["raw_text"])

Loading dataset [sentiment140] from: /kaggle/input/datasets/kazanova/sentiment140/training.1600000.processed.noemoticon.csv
   Applying target sentiment encoding mapping...

Transforming training text data...
Running text normalization pipeline...
Transforming validation text data...
Running text normalization pipeline...


In [16]:
# Grab the specific feature limit from your Twitter config profile (25,000)
tweets_max_feats = SENTIMENT140_CONFIG["preprocessing"]["max_features"]

# --- BM25 Feature Matrices ---
X_tweets_train_matrix, tweets_trained_backbone = vectorize_data(
    X_tweets_train_clean, 
    method="bm25_features", 
    max_features=tweets_max_feats
)

X_tweets_val_matrix = tweets_trained_backbone.transform(X_tweets_val_clean)

tweets_y_train = df_tweets_train["label"].values
tweets_y_val = df_tweets_val["label"].values

# --- TF-IDF Feature Matrices ---
tweets_X_train_tfidf, tweets_trained_vectorizer = vectorize_data(
    X_tweets_train_clean, 
    method="tfidf", 
    max_features=tweets_max_feats
)

tweets_X_val_tfidf = tweets_trained_vectorizer.transform(X_tweets_val_clean)

# 3. View its mathematical type and shapes
print(f"\nType of matrix: {type(tweets_X_train_tfidf)}")
print(f"Matrix shape: {tweets_X_train_tfidf.shape} (Rows: Documents, Columns: Unique Words)")

# 4. View the learned vocabulary words safely
tweets_vocab = tweets_trained_vectorizer.get_feature_names_out()
print(f"Total features learned: {len(tweets_vocab)}")

Vectorizing corpus using method: bm25_features...
Constructing static BM25 feature matrix (max_features=25000)...
Vectorizing corpus using method: tfidf...

Type of matrix: <class 'scipy.sparse._csr.csr_matrix'>
Matrix shape: (1280000, 25000) (Rows: Documents, Columns: Unique Words)
Total features learned: 25000


## Model Training


In [17]:
def train_sentiment_model(df_unified: pd.DataFrame, vectorizer_max_features: int = 10000):
    """
    Trains a Logistic Regression model on a standardized DataFrame 
    containing 'text' and 'label' columns.
    """

    # 1. Train/Validation Split (80% Train, 20% Validation)
    X_train_raw, X_val_raw, y_train, y_val = train_test_split(
        df_unified["text"], 
        df_unified["label"], 
        test_size=0.2, 
        random_state=42, 
        stratify=df_unified["label"]  # Ensures balanced classes in both splits
    )

    # 2. Text Normalization
    normalizer = TextNormalizer(lowercase=True, remove_stopwords=True, method="lemmatization")

    print("Normalizing training data...")
    X_train_clean = normalizer.transform(X_train_raw)
    print("Normalizing validation data...")
    X_val_clean = normalizer.transform(X_val_raw)

    # 3. Vectorization
    X_train_vectors, trained_vectorizer = vectorize_data(
        X_train_clean, method="tfidf", max_features=vectorizer_max_features
    )

    X_val_vectors = trained_vectorizer.transform(X_val_clean)

    # 4. Model Training
    print("Fitting Logistic Regression model...")
    model = LogisticRegression(max_iter=1000, C=1.0)
    model.fit(X_train_vectors, y_train)

    # 5. Evaluation
    predictions = model.predict(X_val_vectors)
    acc = accuracy_score(y_val, predictions)

    print("\n================== EVALUATION REPORT ==================")
    print(f"Validation Accuracy: {acc:.4f}")
    print(classification_report(y_val, predictions))
    
    return model, trained_vectorizer

In [18]:
import os
import pickle
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Assuming these are stored in your src/ folder structure:
# from src.load_data import load_raw_dataset
# from src.preprocess import TextNormalizer
# from src.vectorize import vectorize_data

def run_training_pipeline(data_config: dict, vectorizer_method: str = "tfidf"):
    """
    Runs an end-to-end training and serialization pipeline for a given dataset config.
    """
    dataset_name = data_config["dataset_name"]
    print(f"Starting Training Pipeline for: {dataset_name.upper()}")

    # 1. Load data (Using full data or defined constraints)
    df_raw = load_raw_dataset(data_config)
    
    # 2. Train/Val Split
    df_train, df_val = train_test_split(
        df_raw, 
        test_size=0.2, 
        random_state=42, 
        stratify=df_raw["label"]
    )
    
    # 3. Clean Text using Config Specifications
    normalizer = TextNormalizer(preprocessing_config=data_config["preprocessing"])
    
    print(f"Cleaning training data strings...")
    X_train_clean = normalizer.transform(df_train["raw_text"])
    print(f"Cleaning validation data strings...")
    X_val_clean = normalizer.transform(df_val["raw_text"])
    
    # 4. Vectorize Text
    max_feats = data_config["preprocessing"]["max_features"]
    X_train, trained_backbone = vectorize_data(
        X_train_clean, 
        method=vectorizer_method, 
        max_features=max_feats
    )
    X_val = trained_backbone.transform(X_val_clean)
    
    y_train = df_train["label"].values
    y_val = df_val["label"].values
    
    # 5. Train Core Estimator Model
    print(f"Training Logistic Regression Classifier...")
    model = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
    model.fit(X_train, y_train)
    
    # 6. Evaluate Model Performance
    val_preds = model.predict(X_val)
    acc = accuracy_score(y_val, val_preds)
    
    print(f"\nResults for {dataset_name} ({vectorizer_method}):")
    print(f"   Validation Accuracy: {acc:.4f}")
    print(classification_report(y_val, val_preds))
    
    # 7. Serialize Artifacts to Disk
    os.makedirs("models", exist_ok=True)
    
    model_path = f"models/{dataset_name}_sentiment_model.pkl"
    vectorizer_path = f"models/{dataset_name}_vectorizer.pkl"
    
    with open(model_path, "wb") as f:
        pickle.dump(model, f)
    with open(vectorizer_path, "wb") as f:
        pickle.dump(trained_backbone, f)
        
    print(f"Artifacts successfully saved to 'models/' directory.")
    return acc

In [19]:
# Pass the Twitter configuration profile directly to train
twitter_acc = run_training_pipeline(SENTIMENT140_CONFIG, vectorizer_method="bm25_features")

Starting Training Pipeline for: SENTIMENT140
Loading dataset [sentiment140] from: /kaggle/input/datasets/kazanova/sentiment140/training.1600000.processed.noemoticon.csv
   Applying target sentiment encoding mapping...
Cleaning training data strings...
Running text normalization pipeline...
Cleaning validation data strings...
Running text normalization pipeline...
Vectorizing corpus using method: bm25_features...
Constructing static BM25 feature matrix (max_features=25000)...
Training Logistic Regression Classifier...

Results for sentiment140 (bm25_features):
   Validation Accuracy: 0.7894
              precision    recall  f1-score   support

           0       0.79      0.80      0.79    160000
           1       0.79      0.78      0.79    160000

    accuracy                           0.79    320000
   macro avg       0.79      0.79      0.79    320000
weighted avg       0.79      0.79      0.79    320000

Artifacts successfully saved to 'models/' directory.


In [20]:
# Pass the Amazon configuration profile directly to train
amazon_acc = run_training_pipeline(AMAZON_CONFIG, vectorizer_method="tfidf")

Starting Training Pipeline for: AMAZON
Loading dataset [amazon] from: /kaggle/input/datasets/organizations/snap/amazon-fine-food-reviews/Reviews.csv
   Dropped 42640 ambiguous rows.
   Applying target sentiment encoding mapping...
Cleaning training data strings...
Running text normalization pipeline...
Cleaning validation data strings...
Running text normalization pipeline...
Vectorizing corpus using method: tfidf...
Training Logistic Regression Classifier...

Results for amazon (tfidf):
   Validation Accuracy: 0.9331
              precision    recall  f1-score   support

           0       0.85      0.69      0.76     16407
           1       0.95      0.98      0.96     88756

    accuracy                           0.93    105163
   macro avg       0.90      0.84      0.86    105163
weighted avg       0.93      0.93      0.93    105163

Artifacts successfully saved to 'models/' directory.


## Compare vectorization methods

In [21]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def evaluate_vectorization_methods(X_train_clean, X_val_clean, y_train, y_val, max_features=15000):
    """
    Empirically trains and evaluates BoW, TF-IDF, and BM25 to contrast performance.
    """
    methods = ["bow", "tfidf", "bm25_features"]
    results = []
    
    for method in methods:
        print(f"Running pipeline for method: {method}...")
        
        # 1. Vectorize using your factory function
        X_train_vec, trained_backbone = vectorize_data(
            X_train_clean, method=method, max_features=max_features
        )
        X_val_vec = trained_backbone.transform(X_val_clean)
        
        # 2. Train baseline model
        model = LogisticRegression(max_iter=1000, random_state=42)
        model.fit(X_train_vec, y_train)
        
        # 3. Predict and evaluate
        preds = model.predict(X_val_vec)
        acc = accuracy_score(y_val, preds)
        prec, rec, f1, _ = precision_recall_fscore_support(y_val, preds, average="binary")
        
        results.append({
            "Method": method,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1-Score": f1
        })
        
    # Convert results list to a clean DataFrame for easy comparison
    df_results = pd.DataFrame(results)
    print("\n================ EMPIRICAL PERFORMANCE COMPARISON ================")
    print(df_results.to_string(index=False))
    return df_results

# Execute the evaluation on your Amazon data
# (You can run the same line for your Twitter data variables!)
comparison_df = evaluate_vectorization_methods(
    X_amazon_train_clean, X_amazon_val_clean, amazon_y_train, amazon_y_val, max_features=amazon_max_feats
)

Running pipeline for method: bow...
Vectorizing corpus using method: bow...
Running pipeline for method: tfidf...
Vectorizing corpus using method: tfidf...
Running pipeline for method: bm25_features...
Vectorizing corpus using method: bm25_features...
Constructing static BM25 feature matrix (max_features=15000)...

================ EMPIRICAL PERFORMANCE COMPARISON ================
       Method  Accuracy  Precision   Recall  F1-Score
          bow  0.936242   0.949777 0.976069  0.962744
        tfidf  0.933056   0.945144 0.977410  0.961006
bm25_features  0.912802   0.967889 0.927453  0.947239


In [22]:
comparison_df = evaluate_vectorization_methods(
    X_tweets_train_clean, X_tweets_val_clean, tweets_y_train, tweets_y_val, max_features=tweets_max_feats
)

Running pipeline for method: bow...
Vectorizing corpus using method: bow...
Running pipeline for method: tfidf...
Vectorizing corpus using method: tfidf...
Running pipeline for method: bm25_features...
Vectorizing corpus using method: bm25_features...
Constructing static BM25 feature matrix (max_features=25000)...

================ EMPIRICAL PERFORMANCE COMPARISON ================
       Method  Accuracy  Precision   Recall  F1-Score
          bow  0.798256   0.788948 0.814362  0.801454
        tfidf  0.799687   0.792914 0.811250  0.801977
bm25_features  0.789388   0.793463 0.782444  0.787915


In [23]:
# import pandas as pd
# from lightgbm import LGBMClassifier
# from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# def evaluate_vectorization_methods(X_train_clean, X_val_clean, y_train, y_val, max_features=15000):
#     """
#     Empirically trains and evaluates BoW, TF-IDF, and BM25 to contrast performance.
#     """
#     methods = ["bow", "tfidf", "bm25_features"]
#     results = []
    
#     for method in methods:
#         print(f"Running pipeline for method: {method}...")
        
#         # 1. Vectorize using your factory function
#         X_train_vec, trained_backbone = vectorize_data(
#             X_train_clean, method=method, max_features=max_features
#         )
#         X_val_vec = trained_backbone.transform(X_val_clean)
        
#         # 2. Train baseline model
#         model = LGBMClassifier(n_estimators=300, learning_rate=0.05, random_state=42)
#         model.fit(X_train_vec, y_train)
        
#         # 3. Predict and evaluate
#         preds = model.predict(X_val_vec)
#         acc = accuracy_score(y_val, preds)
#         prec, rec, f1, _ = precision_recall_fscore_support(y_val, preds, average="binary")
        
#         results.append({
#             "Method": method,
#             "Accuracy": acc,
#             "Precision": prec,
#             "Recall": rec,
#             "F1-Score": f1
#         })
        
#     # Convert results list to a clean DataFrame for easy comparison
#     df_results = pd.DataFrame(results)
#     print("\n================ EMPIRICAL PERFORMANCE COMPARISON ================")
#     print(df_results.to_string(index=False))
#     return df_results

# # Execute the evaluation on your Amazon data
# # (You can run the same line for your Twitter data variables!)
# comparison_df = evaluate_vectorization_methods(
#     X_amazon_train_clean, X_amazon_val_clean, amazon_y_train, amazon_y_val, max_features=amazon_max_feats
# )

Running pipeline for method: bow...
Vectorizing corpus using method: bow...


TypeError: Expected np.float32 or np.float64, met type(int64)

In [ ]:
# comparison_df = evaluate_vectorization_methods(
#     X_tweets_train_clean, X_tweets_val_clean, tweets_y_train, tweets_y_val, max_features=tweets_max_feats
# )